# Provenance of validated defaults: recording overrides

Several omicverse methods have defaults that were validated on independent benchmarks (simulated
generators with known truth, labelled public data). Changing such a default because it looked better
on the data at hand (a simulator built from the same data, a development sweep) usually generalises
worse. `ov.utils.validated_defaults` marks these parameters: an override without `override_reason=`
still runs, but emits a `DefaultOverrideWarning`, is logged for the session and is attached to the
result (`DataFrame.attrs['ov_provenance']`, `AnnData.uns['ov_provenance']`, or a dict key), so a report
or a deliverable checker can see it. The idea follows decision records in computational provenance
(W3C PROV-DM).

In [ ]:
import warnings
import numpy as np, pandas as pd
import omicverse as ov
ov.utils.clear_provenance()
rng = np.random.default_rng(0)
n = 3000
X = pd.DataFrame((rng.random((n, 200)) < 0.3).astype(float), columns=[f'g{i}' for i in range(200)])
y = (rng.random(n) < 1 / (1 + np.exp(-(-2 + 2.5 * X.g0 + 2.0 * X.g1)))).astype(int)

## 1. Defaults kept: nothing is recorded

In [ ]:
res = ov.genetics.trait_determinants(X, y, modifier=None)
print(sorted(res.feature[res.is_determinant]))
print(ov.utils.provenance_report())

['g0', 'g1']
ov provenance: all validated defaults kept.


## 2. Overrides: without a reason they warn and are attached to the result; with `override_reason=` they are only logged

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    res2 = ov.genetics.trait_determinants(X, y, modifier=None, n_pcs=0, select='bic')
print([str(x.message)[:120] for x in w if issubclass(x.category, ov.utils.DefaultOverrideWarning)])
print(res2.attrs['ov_provenance'])
res3 = ov.genetics.trait_determinants(X, y, modifier=None, n_pcs=0,
                                      override_reason='no population structure in this simulated cohort')
print(ov.utils.provenance_report())

["trait_determinants: validated default(s) overridden without override_reason: select='bic' (default 'ebic'), n_pcs=0 (def"]
[{'function': 'omicverse.genetics._determinants.trait_determinants', 'parameter': 'select', 'default': 'ebic', 'value': 'bic', 'reason': None, 'time': '2026-11-23T14:20:04'}, {'function': 'omicverse.genetics._determinants.trait_determinants', 'parameter': 'n_pcs', 'default': 10, 'value': 0, 'reason': None, 'time': '2026-11-23T14:20:04'}]
ov provenance: 2 unexplained / 3 overrides
  omicverse.genetics._determinants.trait_determinants: select='bic' (default 'ebic') -- NO REASON GIVEN
  omicverse.genetics._determinants.trait_determinants: n_pcs=0 (default 10) -- NO REASON GIVEN
  omicverse.genetics._determinants.trait_determinants: n_pcs=0 (default 10) -- no population structure in this simulated cohort


## 3. In a deliverable check

`ov.utils.write_provenance(path)` writes the session log as JSON next to a deliverable; every line marked
**NO REASON GIVEN** should be reverted to the default or justified with independent evidence.

In [ ]:
import tempfile, os, json
p = ov.utils.write_provenance(os.path.join(tempfile.mkdtemp(), 'provenance.json'))
print(json.load(open(p))['overrides'][0]['parameter'], '...', len(json.load(open(p))['overrides']), 'records')

select ... 3 records


## 4. Validated outputs and the final deliverable

An override of an *argument* is caught by the decorator. A replaced *result* is not: the agent calls the
function with its defaults and then rewrites the labels by hand. `ov.utils.record_output` stores a validated
function's output in the run-local ledger (`$OMICOS_PROVENANCE_DIR`, else `.omicos/provenance/`), and
`ov.utils.check_deliverable(file)` reports, column by column, how far the delivered file still agrees with it.
A departure without a declared reason is **UNEXPLAINED**.

In [5]:
import os, tempfile
import numpy as np, pandas as pd
import omicverse as ov
work = tempfile.mkdtemp()
os.environ['OV_PROVENANCE_DIR'] = os.path.join(work, 'prov')
rng = np.random.default_rng(1)
lib = pd.DataFrame({'sample_id': [f's{i}' for i in range(200)],
                    'diagnosis': rng.choice(['cd', 'uc', 'nonibd'], 200)})
ov.utils.record_output('omicverse.demo.annotate', lib, keys=['sample_id'], values=['diagnosis'])
lib.to_csv(os.path.join(work, 'deliverable.csv'), index=False)
print(ov.utils.check_deliverable(os.path.join(work, 'deliverable.csv')).attrs['summary'])

check_deliverable(/tmp/tmpc3denmwc/deliverable.csv): 1 column(s) traced to validated outputs, 1 match, 0 UNEXPLAINED departure(s).
  annotate:diagnosis agreement 100.0% over 200 rows -> match


In [6]:
hand = lib.copy()
hand.loc[hand.index[:40], 'diagnosis'] = 'nonibd'          # a hand rule replaces 40 labels
hand.to_csv(os.path.join(work, 'deliverable.csv'), index=False)
res = ov.utils.check_deliverable(os.path.join(work, 'deliverable.csv'))
print(res.attrs['summary'])

check_deliverable(/tmp/tmpc3denmwc/deliverable.csv): 1 column(s) traced to validated outputs, 0 match, 1 UNEXPLAINED departure(s).
  annotate:diagnosis agreement 86.5% over 200 rows -> UNEXPLAINED
  Deliver the validated output for these columns, or call ov.utils.declare_departure(function, reason) naming independent evidence (data not used to tune the replacement).


## 5. Declaring a departure

`declare_departure(function, reason)` records why the deliverable differs. The reason must name independent
evidence (data that was not used to build the replacement); the check then reports `departs` with that reason
instead of UNEXPLAINED.

In [7]:
ov.utils.declare_departure('annotate', 'chart review of 40 samples by a second reader (not used to fit anything)',
                           columns=['diagnosis'])
print(ov.utils.check_deliverable(os.path.join(work, 'deliverable.csv')).attrs['summary'])

check_deliverable(/tmp/tmpc3denmwc/deliverable.csv): 1 column(s) traced to validated outputs, 0 match, 0 UNEXPLAINED departure(s).
  annotate:diagnosis agreement 86.5% over 200 rows -> departs (chart review of 40 samples by a second reader (not used to fit anything))


## 6. Auditing a replacement on the default's own protocol

`compare_to_default` refits default and override on the same leave-one-group-out splits (batch / cohort:
the unit the result must generalise across) and accepts the override only with a bootstrap improvement after
a Bonferroni correction for every alternative tried, no collapse of the worst group, and no stronger
association with a technical confounder than the labels themselves have. Below the override reads the batch,
which a random hold-out would reward.

In [8]:
from sklearn.linear_model import LogisticRegression
n = 600
batch = rng.choice(['b1', 'b2', 'b3', 'b4'], n)
y = rng.choice(['case', 'ctrl'], n)
signal = (y == 'case') * 1.0 + rng.normal(0, 1, n)
batch_shift = np.where(batch == 'b1', 2.0, 0.0)
X = np.c_[signal, batch_shift + rng.normal(0, 0.1, n)]
y = np.where(batch == 'b1', 'case', y)                        # batch b1 is all cases: a confound

def default(tr, te):
    return LogisticRegression().fit(X[tr, :1], y[tr]).predict(X[te, :1])

def override(tr, te):                                          # uses the batch-coupled feature
    return LogisticRegression().fit(X[tr], y[tr]).predict(X[te])

audit = ov.utils.compare_to_default(y, default, override, groups=batch, confounder=batch,
                                    n_candidates=3, function='demo.classify')
print(audit['verdict'], audit['reasons'])

keep_default ['collapse: worst held-out group accuracy 0.243 vs default 0.467']
